# 3. Structured data first

Metadata extracted from `episode.yaml` is knowledge too. Many questions need no vector search and no full-text search: plain SQL answers them exactly.

In [ ]:
from demo.notebook import embed_one, query

## Part A: no search at all

The latest 3 episodes: a sort, not a similarity.

In [ ]:
latest = query("""
SELECT TOP (3) episode_number, title, publication_date
FROM dbo.Episodes
ORDER BY publication_date DESC
""")
display(latest)
assert latest.episode_number[0] == 182, "the latest episode by date is #182"

The chapters of episode 164 with timestamps, and its links.

In [ ]:
chapters = query("""
SELECT position, start_timestamp, title
FROM dbo.EpisodeChapters
WHERE episode_number = 164
ORDER BY position
""")
display(chapters)
links = query("""
SELECT l.title, l.url
FROM dbo.EpisodeLinks AS el
JOIN dbo.Links AS l ON l.link_slug = el.link_slug
WHERE el.episode_number = 164
ORDER BY el.position
""")
display(links)
assert len(chapters) > 0 and len(links) > 0, "episode 164 has chapters and links"

Episodes with a guest: a join, not a search.

In [ ]:
guests = query("""
SELECT e.episode_number, e.title, p.name AS guest
FROM dbo.Episodes AS e
JOIN dbo.EpisodeGuests AS g ON g.episode_number = e.episode_number
JOIN dbo.People AS p ON p.person_slug = g.person_slug
ORDER BY e.publication_date
""")
display(guests)
assert set(guests.episode_number) == {144, 169}, "only episodes 144 and 169 have guests"

Episodes per tag in 2025: a `GROUP BY`.

In [ ]:
tags = query("""
SELECT TOP (10) t.tag, COUNT(*) AS episodes
FROM dbo.EpisodeTags AS t
JOIN dbo.Episodes AS e ON e.episode_number = t.episode_number
WHERE e.publication_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY t.tag
ORDER BY episodes DESC, t.tag
""")
display(tags)
assert len(tags) > 0, "tags of 2025 episodes exist"

## Contrast: "najnowszy odcinek"

A vector search has no idea of time: neither the title embedding nor the turn search returns the latest episode. `ORDER BY publication_date DESC` does.

In [ ]:
question = "najnowszy odcinek"
vector = embed_one(question)
by_title = query("""
SELECT TOP (1) episode_number, title, publication_date,
    VECTOR_DISTANCE('cosine', title_embedding, CAST(? AS vector(1536))) AS distance
FROM dbo.Episodes
ORDER BY distance
""", (vector,))
by_turns = query("""
DECLARE @embedding vector(1536) = CAST(? AS vector(1536));
EXEC dbo.SearchChunks @searchText = ?, @embedding = @embedding, @topK = 1;
""", (vector, question))
by_date = query("SELECT TOP (1) episode_number, title, publication_date FROM dbo.Episodes ORDER BY publication_date DESC")
display(by_title, by_turns[["episode_number", "title", "publication_date", "transcript_text"]], by_date)
assert by_date.episode_number[0] == 182, "ORDER BY returns the latest episode, #182"
assert by_title.episode_number[0] != 182, "the title vector search does not return the latest episode"
assert by_turns.episode_number[0] != 182, "the turn search does not return the latest episode"

## Part B: search plus structure

"What did we say about Kubernetes in 2025, and with whom?" The search finds the turns; joins add the episode, the speakers and the guests.

In [ ]:
kubernetes = query("""
SET NOCOUNT ON;
DECLARE @embedding vector(1536) = CAST(? AS vector(1536));
DROP TABLE IF EXISTS #hits;
CREATE TABLE #hits (
    episode_number INT, title NVARCHAR(400), publication_date DATE, chunk_number INT,
    speaker NVARCHAR(100), start_timestamp VARCHAR(12), transcript_text NVARCHAR(MAX),
    score FLOAT, rank_fulltext INT, rank_phrase INT, rank_vector INT);

INSERT INTO #hits
EXEC dbo.SearchChunks @searchText = ?, @embedding = @embedding, @topK = 10,
    @fromDate = '2025-01-01', @toDate = '2025-12-31';

SELECT
    e.episode_number, e.title, e.publication_date,
    COUNT(*) AS matching_turns,
    MIN(h.rank_fulltext) AS best_fulltext_rank,
    (SELECT STRING_AGG(s.speaker, ', ')
     FROM (SELECT DISTINCT speaker FROM #hits WHERE episode_number = e.episode_number) AS s) AS speakers,
    (SELECT STRING_AGG(p.name, ', ')
     FROM dbo.EpisodeGuests AS g JOIN dbo.People AS p ON p.person_slug = g.person_slug
     WHERE g.episode_number = e.episode_number) AS guests
FROM #hits AS h
JOIN dbo.Episodes AS e ON e.episode_number = h.episode_number
GROUP BY e.episode_number, e.title, e.publication_date
ORDER BY MAX(h.score) DESC;
""", (embed_one("Kubernetes"), "Kubernetes"))
display(kubernetes)
assert 148 in set(kubernetes.episode_number), "episode 148 (Kubernetes Security) is among the hits"
assert kubernetes.best_fulltext_rank.notna().any(), "full-text found the word, not only the vector leg"
assert kubernetes.publication_date.astype(str).between("2025-01-01", "2025-12-31").all(), "only 2025"

## Part C: counting

"How often did we talk about Kubernetes in 2025?" is an aggregate over all turns, not a top-k list.

In [ ]:
mentions = query("EXEC dbo.CountMentions ?, ?, ?", ("Kubernetes", "2025-01-01", "2025-12-31"))
print(f"{mentions.mention_count.sum()} turns in {len(mentions)} episodes")
display(mentions)
assert len(mentions) >= 1, "Kubernetes is mentioned in 2025"
assert mentions.publication_date.astype(str).between("2025-01-01", "2025-12-31").all(), "only 2025"

## Question type → what answers it

| Question type | Example | What answers it |
|---|---|---|
| Lookup by key | the chapters of #164 | `WHERE episode_number = 164` |
| List / sort | the latest 3 episodes | `ORDER BY publication_date DESC` |
| Relation | episodes with a guest | `JOIN EpisodeGuests, People` |
| Count | how often Kubernetes in 2025 | `dbo.CountMentions` (full-text, `COUNT`) |
| Exact phrase | "vendor lock-in" | `CONTAINSTABLE` with a phrase |
| Meaning | where to host a side project cheaply | `VECTOR_DISTANCE` / `dbo.SearchChunks` |

These are the same shapes an agent gets as tools: one tool per question type, each a plain SQL query.